# 🌍 Multilingual Toxicity Detection System — Final
**Pipeline:** Input Text → Language Detection → Translation (if needed) → Detoxify Model → Multi-Feature Output

| Stage | Tool |
|---|---|
| Language Detection | `langdetect` |
| Translation | `deep_translator` (Google) |
| Toxicity Model | `detoxify` multilingual (XLM-RoBERTa) |
| Output | 7 toxicity labels + severity + JSON + visualization |


In [ ]:
import subprocess, sys
pkgs = ["detoxify", "langdetect", "deep-translator", "pandas", "numpy",
        "torch", "transformers", "matplotlib", "seaborn", "tqdm"]
for p in pkgs:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p])
print("✅ All libraries installed!")


In [ ]:
import os, json, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Any, Optional

# Core NLP
from langdetect import detect, detect_langs, LangDetectException
from deep_translator import GoogleTranslator
from detoxify import Detoxify

import torch
from tqdm import tqdm

warnings.filterwarnings('ignore')
np.random.seed(42)

print("✅ All imports successful!")
print(f"PyTorch: {torch.__version__} | Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")


In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
RESULTS_DIR = Path("d:/NLP/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Languages supported NATIVELY by Detoxify multilingual (no translation needed)
NATIVE_LANGUAGES = {'en', 'fr', 'es', 'it', 'pt', 'tr', 'ru'}

# Full language map for display
LANGUAGE_NAMES = {
    'en': 'English',    'es': 'Spanish',    'fr': 'French',
    'de': 'German',     'it': 'Italian',    'pt': 'Portuguese',
    'ru': 'Russian',    'ja': 'Japanese',   'zh-cn': 'Chinese (Simplified)',
    'ar': 'Arabic',     'nl': 'Dutch',      'pl': 'Polish',
    'tr': 'Turkish',    'ko': 'Korean',     'vi': 'Vietnamese',
    'hi': 'Hindi',      'kn': 'Kannada',    'ta': 'Tamil',
    'te': 'Telugu',     'ml': 'Malayalam',  'bn': 'Bengali',
    'mr': 'Marathi',    'gu': 'Gujarati',   'ur': 'Urdu',
    'id': 'Indonesian', 'th': 'Thai',
}

# Severity thresholds
SEVERITY_LEVELS = [
    (0.80, '🔴 CRITICAL'),
    (0.60, '🟠 HIGH'),
    (0.40, '🟡 MEDIUM'),
    (0.20, '🔵 LOW'),
    (0.00, '🟢 NONE'),
]

print("✅ Configuration loaded")
print(f"  Native Detoxify languages: {sorted(NATIVE_LANGUAGES)}")
print(f"  Results will be saved to: {RESULTS_DIR}")


In [ ]:
class LanguageDetector:
    """Detect language of input text using langdetect."""

    def detect(self, text: str) -> Dict[str, Any]:
        if not text or len(text.strip()) < 3:
            return {'code': 'unknown', 'name': 'Unknown', 'confidence': 0.0,
                    'is_native': False, 'needs_translation': False}
        try:
            probs = detect_langs(text)
            top = probs[0]
            code = top.lang
            return {
                'code': code,
                'name': LANGUAGE_NAMES.get(code, code.upper()),
                'confidence': round(top.prob, 3),
                'all_probs': [(p.lang, round(p.prob, 3)) for p in probs[:3]],
                'is_native': code in NATIVE_LANGUAGES,
                'needs_translation': code not in NATIVE_LANGUAGES,
            }
        except LangDetectException as e:
            return {'code': 'error', 'name': 'Error', 'confidence': 0.0,
                    'is_native': False, 'needs_translation': False, 'error': str(e)}

lang_detector = LanguageDetector()
print("✅ Language Detector initialized")

# Quick test
for sample in ["Hello world", "यह एक परीक्षण है", "Este es un comentario"]:
    r = lang_detector.detect(sample)
    print(f"  '{sample[:30]}' → {r['name']} ({r['code']}) confidence={r['confidence']:.0%} | native={r['is_native']}")


In [ ]:
class TextTranslator:
    """Translate text to English using deep_translator (Google)."""

    def __init__(self):
        self._cache: Dict[tuple, str] = {}
        self.stats = {'translated': 0, 'cache_hits': 0, 'skipped': 0}

    def translate(self, text: str, src_lang: str) -> Dict[str, Any]:
        # If already a native language, skip translation
        if src_lang in NATIVE_LANGUAGES:
            self.stats['skipped'] += 1
            return {'translated': False, 'original': text, 'result': text, 'src_lang': src_lang}

        cache_key = (text[:150], src_lang)
        if cache_key in self._cache:
            self.stats['cache_hits'] += 1
            return {'translated': True, 'original': text, 'result': self._cache[cache_key],
                    'src_lang': src_lang, 'cached': True}
        try:
            translated = GoogleTranslator(source=src_lang, target='en').translate(text)
            self._cache[cache_key] = translated
            self.stats['translated'] += 1
            return {'translated': True, 'original': text, 'result': translated,
                    'src_lang': src_lang, 'cached': False}
        except Exception as e:
            # Fallback: return original if translation fails
            return {'translated': False, 'original': text, 'result': text,
                    'src_lang': src_lang, 'error': str(e)}

translator = TextTranslator()
print("✅ Translator initialized (deep_translator / Google)")


In [ ]:
print("=" * 70)
print("🔄 TRANSLATION VERIFICATION TEST")
print("=" * 70)

verification_cases = [
    # (lang_code, text, expected_behaviour)
    ("hi", "यह एक बहुत अच्छा उत्पाद है",         "Hindi  → TRANSLATE"),
    ("hi", "तुम बिल्कुल बेकार हो, मैं तुमसे नफरत करता हूं!", "Hindi  → TRANSLATE (toxic)"),
    ("hi", "आज मौसम बहुत अच्छा है",             "Hindi  → TRANSLATE (clean)"),
    ("de", "Du bist ein kompletter Idiot!",        "German → TRANSLATE"),
    ("ar", "أنت أحمق تماماً وأنا أكرهك!",          "Arabic → TRANSLATE"),
    ("es", "Hola, ¿cómo estás?",                  "Spanish→ SKIP (native)"),
    ("en", "The weather is nice today.",           "English→ SKIP (native)"),
    ("ru", "Ты полный идиот!",                     "Russian→ SKIP (native)"),
]

pass_count = 0
for lang, text, label in verification_cases:
    r = translator.translate(text, lang)
    was_translated = r['translated']
    en_text = r['result']
    status = "✅" if ("SKIP" in label) == (not was_translated) else "❌"
    if status == "✅":
        pass_count += 1
    action = "TRANSLATED" if was_translated else "SKIPPED (native)"
    print(f"{status} [{label:28s}] {action}")
    print(f"     IN : {text[:60]}")
    print(f"     OUT: {en_text[:60]}")
    print()

print(f"{'='*70}")
print(f"Translation verification: {pass_count}/{len(verification_cases)} passed")
print(f"Cache stats: {translator.stats}")


In [ ]:
class TextPreprocessor:
    """Clean and normalize text before model inference."""

    def preprocess(self, text: str) -> str:
        if not text:
            return ""
        text = re.sub(r'http\S+|www\.\S+', '', text)   # Remove URLs
        text = re.sub(r'\S+@\S+', '', text)              # Remove emails
        text = re.sub(r'\s+', ' ', text).strip()          # Normalize spaces
        return text

preprocessor = TextPreprocessor()
print("✅ Text Preprocessor initialized")


In [ ]:
print("=" * 70)
print("🤖 LOADING DETOXIFY MULTILINGUAL MODEL")
print("=" * 70)
print("  Model: unitary/multilingual-toxic-xlm-roberta")
print("  Downloading on first run (~1 GB)... please wait\n")

try:
    detoxify_model = Detoxify('multilingual')
    # Get the actual label names from a test prediction
    _test_preds = detoxify_model.predict("test")
    DETOXIFY_LABELS = list(_test_preds.keys())
    print(f"✅ Model loaded successfully!")
    print(f"  Labels: {DETOXIFY_LABELS}")
except Exception as e:
    print(f"❌ Error: {e}")
    detoxify_model = None
    DETOXIFY_LABELS = ['toxicity', 'severe_toxicity', 'obscene', 'threat', 'insult', 'identity_attack', 'sexual_explicit']

print("=" * 70)


In [ ]:
def get_severity(max_score: float) -> str:
    for threshold, level in SEVERITY_LEVELS:
        if max_score >= threshold:
            return level
    return '🟢 NONE'

def get_category_emoji(label: str) -> str:
    return {
        'toxicity': '☠️', 'severe_toxicity': '💀', 'obscene': '🤬',
        'threat': '⚠️', 'insult': '😤', 'identity_attack': '🎯',
        'sexual_explicit': '🔞',
    }.get(label, '❓')

class MultilingualToxicityPipeline:
    """
    Complete pipeline:
      1. Language Detection
      2. Translation (if language not natively supported)
      3. Text Preprocessing
      4. Detoxify Model Inference
      5. Multi-feature structured output
    """

    def __init__(self, model, detector, translator, preprocessor):
        self.model = model
        self.detector = detector
        self.translator = translator
        self.preprocessor = preprocessor

    def process(self, text: str, threshold: float = 0.5) -> Dict[str, Any]:
        # Step 1: Detect language
        lang_info = self.detector.detect(text)

        # Step 2: Translate if needed
        trans_info = self.translator.translate(text, lang_info['code'])
        text_for_model = trans_info['result']

        # Step 3: Preprocess
        clean_text = self.preprocessor.preprocess(text_for_model)
        if not clean_text:
            clean_text = text_for_model

        # Step 4: Model inference
        raw_preds = self.model.predict(clean_text)

        # Step 5: Structure output
        scores = {k: float(v) for k, v in raw_preds.items()}
        max_score = max(scores.values())
        detected_types = [k for k, v in scores.items() if v >= threshold]
        severity = get_severity(max_score)

        return {
            'input': {
                'original_text': text,
                'char_count': len(text),
                'word_count': len(text.split()),
            },
            'language': {
                'detected_code': lang_info['code'],
                'detected_name': lang_info['name'],
                'confidence': lang_info['confidence'],
                'is_native_to_model': lang_info['is_native'],
                'was_translated': trans_info['translated'],
                'translated_text': trans_info['result'] if trans_info['translated'] else None,
            },
            'preprocessed_text': clean_text,
            'toxicity': {
                'is_toxic': max_score >= threshold,
                'severity': severity,
                'max_score': round(max_score, 4),
                'threshold_used': threshold,
                'detected_types': detected_types,
                'num_types_detected': len(detected_types),
            },
            'label_scores': {k: round(v, 4) for k, v in scores.items()},
            'timestamp': datetime.now().isoformat(),
        }

    def process_batch(self, texts: List[str], threshold: float = 0.5) -> List[Dict]:
        results = []
        for text in tqdm(texts, desc="Processing"):
            results.append(self.process(text, threshold))
        return results

pipeline = MultilingualToxicityPipeline(detoxify_model, lang_detector, translator, preprocessor)
print("✅ Multilingual Toxicity Pipeline initialized and ready!")


In [ ]:
print("=" * 70)
print("🌍 MULTILINGUAL TOXICITY TEST")
print("=" * 70)

test_inputs = [
    # ── Natively supported by Detoxify (no translation needed) ──
    ("English",    "You are an absolute idiot and I hate you!"),
    ("Spanish",    "¡Eres un completo idiota y te odio!"),
    ("French",     "Tu es un idiot et je te déteste!"),
    ("Russian",    "Ты полный идиот, я тебя ненавижу!"),
    ("Italian",    "Sei un idiota completo e ti odio!"),
    ("Portuguese", "Você é um idiota completo e eu te odeio!"),
    ("Turkish",    "Sen tam bir aptalsin ve senden nefret ediyorum!"),
    # ── Hindi — requires translation to English first ──
    ("Hindi",      "तुम बिल्कुल बेकार और घटिया इंसान हो!"),
    ("Hindi",      "तुम्हें यहाँ से चले जाना चाहिए, कोई तुम्हें पसंद नहीं करता!"),
    ("Hindi",      "आज मौसम बहुत अच्छा है, मुझे गर्मी पसंद है!"),
    ("Hindi",      "यह फिल्म बहुत अच्छी थी, मुझे पसंद आई।"),
    # ── Other languages needing translation ──
    ("German",     "Du bist ein kompletter Idiot und ich hasse dich!"),
    ("Arabic",     "أنت أحمق تماماً وأنا أكرهك!"),
    ("Kannada",    "ನೀವು ಸಂಪೂರ್ಣ ಮೂರ್ಖರು ಮತ್ತು ನಾನು ನಿಮ್ಮನ್ನು ದ್ವೇಷಿಸುತ್ತೇನೆ!"),
    # ── Non-toxic ──
    ("English",    "The weather is beautiful today, I love summer!"),
    ("Hindi",      "भारत एक महान देश है और यहाँ की संस्कृति अद्भुत है।"),
]

results = []
for lang_label, text in test_inputs:
    r = pipeline.process(text)
    results.append(r)
    
    lang_flag = "🌐" if r['language']['was_translated'] else "✅"
    toxic_icon = "⚠️ TOXIC" if r['toxicity']['is_toxic'] else "✅ CLEAN"
    
    print(f"\n{'─'*68}")
    print(f"  [{lang_label:12s}] {lang_flag} → {toxic_icon} | {r['toxicity']['severity']}")
    print(f"  Text    : {text[:65]}")
    if r['language']['was_translated']:
        print(f"  English : {r['translated_text'] or r['preprocessed_text']}")
    print(f"  Scores  : ", end="")
    for label, score in r['label_scores'].items():
        if score > 0.05:
            emoji = get_category_emoji(label)
            print(f"{emoji}{label}={score:.2f} ", end="")
    print()

print(f"\n{'='*70}")
print(f"✅ Tested {len(results)} samples | Native: {sum(1 for r in results if r['language']['is_native_to_model'])} | Translated: {sum(1 for r in results if r['language']['was_translated'])}")


In [ ]:
def visualize_results(results: List[Dict], title: str = "Toxicity Analysis Results"):
    """Generate a comprehensive multi-panel visualization."""
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.patch.set_facecolor('#0f0f1a')
    fig.suptitle(title, fontsize=16, fontweight='bold', color='white', y=0.98)
    
    for ax in axes.flatten():
        ax.set_facecolor('#1a1a2e')
        for spine in ax.spines.values():
            spine.set_color('#444')
    
    labels = [r['label_scores'] for r in results]
    all_label_keys = list(labels[0].keys()) if labels else DETOXIFY_LABELS
    texts_short = [r['input']['original_text'][:30] + "..." for r in results]
    
    # ── Panel 1: Heatmap of label scores ─────────────────────────────────────
    ax1 = axes[0, 0]
    score_matrix = np.array([[r['label_scores'].get(k, 0) for k in all_label_keys] for r in results])
    im = ax1.imshow(score_matrix, cmap='RdYlGn_r', vmin=0, vmax=1, aspect='auto')
    ax1.set_xticks(range(len(all_label_keys)))
    ax1.set_xticklabels([k.replace('_', '\n') for k in all_label_keys], color='white', fontsize=8)
    ax1.set_yticks(range(len(texts_short)))
    ax1.set_yticklabels(texts_short, color='white', fontsize=7)
    ax1.set_title('Label Score Heatmap', color='white', fontweight='bold')
    plt.colorbar(im, ax=ax1)
    for i in range(score_matrix.shape[0]):
        for j in range(score_matrix.shape[1]):
            ax1.text(j, i, f'{score_matrix[i, j]:.2f}', ha='center', va='center',
                     color='white', fontsize=6, fontweight='bold')
    
    # ── Panel 2: Max toxicity score per sample ────────────────────────────────
    ax2 = axes[0, 1]
    max_scores = [r['toxicity']['max_score'] for r in results]
    colors = ['#e74c3c' if s >= 0.5 else '#2ecc71' for s in max_scores]
    bars = ax2.barh(range(len(results)), max_scores, color=colors, edgecolor='#333', height=0.7)
    ax2.set_yticks(range(len(results)))
    ax2.set_yticklabels(texts_short, color='white', fontsize=7)
    ax2.set_xlabel('Max Toxicity Score', color='white')
    ax2.set_title('Max Toxicity Score Per Sample', color='white', fontweight='bold')
    ax2.axvline(0.5, color='yellow', linestyle='--', alpha=0.7, label='Threshold (0.5)')
    ax2.legend(facecolor='#1a1a2e', labelcolor='white')
    ax2.tick_params(colors='white')
    ax2.set_xlim(0, 1)
    for bar, score in zip(bars, max_scores):
        ax2.text(min(score + 0.02, 0.97), bar.get_y() + bar.get_height()/2,
                 f'{score:.3f}', va='center', color='white', fontsize=8)
    
    # ── Panel 3: Toxicity type distribution (pie chart) ───────────────────────
    ax3 = axes[1, 0]
    type_counts = {}
    for r in results:
        for t in r['toxicity']['detected_types']:
            type_counts[t] = type_counts.get(t, 0) + 1
    
    if type_counts:
        palette = ['#e74c3c','#e67e22','#f39c12','#9b59b6','#3498db','#1abc9c','#e91e63']
        wedges, texts_pie, autotexts = ax3.pie(
            type_counts.values(), labels=type_counts.keys(),
            autopct='%1.0f%%', colors=palette[:len(type_counts)],
            textprops={'color': 'white', 'fontsize': 9}
        )
        for at in autotexts:
            at.set_color('white')
    else:
        ax3.text(0.5, 0.5, 'No Toxic Content\nDetected', ha='center', va='center',
                 color='#2ecc71', fontsize=14, transform=ax3.transAxes)
    ax3.set_title('Detected Toxicity Type Distribution', color='white', fontweight='bold')
    
    # ── Panel 4: Language & translation stats ─────────────────────────────────
    ax4 = axes[1, 1]
    lang_counts = {}
    for r in results:
        lang = r['language']['detected_name']
        lang_counts[lang] = lang_counts.get(lang, 0) + 1
    
    native_count = sum(1 for r in results if r['language']['is_native_to_model'])
    translated_count = sum(1 for r in results if r['language']['was_translated'])
    toxic_count = sum(1 for r in results if r['toxicity']['is_toxic'])
    clean_count = len(results) - toxic_count

    categories = ['Native\n(no transl.)', 'Translated', 'Toxic', 'Clean']
    values = [native_count, translated_count, toxic_count, clean_count]
    bar_colors = ['#3498db', '#f39c12', '#e74c3c', '#2ecc71']
    bars4 = ax4.bar(categories, values, color=bar_colors, edgecolor='#333', width=0.6)
    ax4.set_title('Processing & Result Stats', color='white', fontweight='bold')
    ax4.set_ylabel('Count', color='white')
    ax4.tick_params(colors='white')
    for bar, val in zip(bars4, values):
        ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 str(val), ha='center', color='white', fontsize=12, fontweight='bold')
    ax4.set_ylim(0, max(values) + 2)

    plt.tight_layout()
    plot_path = RESULTS_DIR / "toxicity_analysis.png"
    plt.savefig(plot_path, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()
    print(f"\n✅ Plot saved to: {plot_path}")

visualize_results(results, "🌍 Multilingual Toxicity Detection — Analysis Report")


In [ ]:
def print_detailed_report(result: Dict):
    """Print a rich, structured report for a single result."""
    t = result['toxicity']
    l = result['language']
    
    print("┌" + "─"*68 + "┐")
    print(f"│ {'TOXICITY ANALYSIS REPORT':^68} │")
    print("├" + "─"*68 + "┤")
    print(f"│ 📝 Input  : {result['input']['original_text'][:55]:<55} │")
    print(f"│ 🌍 Language: {l['detected_name']:<10} ({l['detected_code']}) | Confidence: {l['confidence']:.0%}{'':>10} │")
    if l['was_translated']:
        print(f"│ 🔄 Translated: {result['preprocessed_text'][:52]:<52} │")
    else:
        print(f"│ ✅ Native  : No translation needed{'':>35} │")
    print("├" + "─"*68 + "┤")
    print(f"│ {'IS TOXIC':<12}: {'⚠️  YES' if t['is_toxic'] else '✅  NO':<56} │")
    print(f"│ {'SEVERITY':<12}: {t['severity']:<56} │")
    print(f"│ {'MAX SCORE':<12}: {t['max_score']:<56.4f} │")
    if t['detected_types']:
        print(f"│ {'TYPES':<12}: {', '.join(t['detected_types']):<56} │")
    print("├" + "─"*68 + "┤")
    print(f"│ {'LABEL':<22} {'SCORE':>8}  {'BAR':>30}     │")
    print("│" + "─"*68 + "│")
    for label, score in result['label_scores'].items():
        bar_len = int(score * 30)
        bar = '█' * bar_len + '░' * (30 - bar_len)
        emoji = get_category_emoji(label)
        print(f"│ {emoji} {label:<20} {score:>8.4f}  {bar}     │")
    print("└" + "─"*68 + "┘")

# Show detailed reports for the first 3 results
print("=" * 70)
print("📊 DETAILED MULTI-FEATURE OUTPUT REPORTS")
print("=" * 70)
for r in results[:3]:
    print()
    print_detailed_report(r)


In [ ]:
# Export all results to JSON
output_path = RESULTS_DIR / f"toxicity_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"✅ Results exported to: {output_path}")
print(f"   Total samples analysed : {len(results)}")
print(f"   Toxic detected         : {sum(1 for r in results if r['toxicity']['is_toxic'])}")
print(f"   Clean detected         : {sum(1 for r in results if not r['toxicity']['is_toxic'])}")
print(f"   Languages translated   : {sum(1 for r in results if r['language']['was_translated'])}")

# Print a summary table
print("\n" + "=" * 70)
print("SUMMARY TABLE")
print("=" * 70)
print(f"{'#':>3} | {'Language':^12} | {'Trans':^5} | {'Toxic':^5} | {'Severity':^15} | {'Max Score':^9}")
print("─" * 70)
for i, r in enumerate(results, 1):
    lang = r['language']['detected_name'][:10]
    trans = '✓' if r['language']['was_translated'] else '─'
    toxic = '⚠️ YES' if r['toxicity']['is_toxic'] else '✅ NO '
    sev = r['toxicity']['severity'].replace('🔴','').replace('🟠','').replace('🟡','').replace('🔵','').replace('🟢','').strip()
    score = r['toxicity']['max_score']
    print(f"{i:>3} | {lang:^12} | {trans:^5} | {toxic:^5} | {sev:^15} | {score:^9.4f}")


In [ ]:
# ── INTERACTIVE: Analyse any text ─────────────────────────────────────────────
def analyse(text: str, threshold: float = 0.5):
    """Analyse any text for toxicity. Works for any language."""
    r = pipeline.process(text, threshold)
    print_detailed_report(r)
    return r

# ✏️  Change the text below to test any input!
my_text = "Write your own text here to test the pipeline!"
result = analyse(my_text)
